# Lab Assignment 3 - Latent Space Classification
# Notebook 1: Train Convolutional Autoencoder

**Authors:**
- Patricia Guadalupe Alvarenga Mairena
- Francisco Manuel Vázquez Fernández

## 1. Import Libraries and Setup

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
    # Reduces the number of warnings
    
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds

# Configure matplotlib for inline plotting
%matplotlib inline

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("="*60)
print("STL-10 Dataset CNN Classifier")
print("="*60)
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("="*60)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
# To avoid stalling due to memory problems

/home/leandro/miniconda3/envs/apaubio/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


STL-10 Dataset CNN Classifier
TensorFlow version: 2.20.0
Keras version: 3.13.1
GPU Available: True


## 2. Configuration parameters

In [ ]:
IMG_SIZE = 96  # STL-10 native size
BATCH_SIZE = 128
EPOCHS = 200
NUM_CLASSES = 10  # airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck
INITIAL_LR = 0.001
LATENT_DIM = 256  # Size of the bottleneck representation

# Class names for visualization
CLASS_NAMES = ['airplane', 'bird', 'car', 'cat', 'deer', 
               'dog', 'horse', 'monkey', 'ship', 'truck']

print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Number of Classes: {NUM_CLASSES}")
print(f"Initial Learning Rate: {INITIAL_LR}")
print(f"Latent Dimension: {LATENT_DIM}")
print(f"\nClasses: {', '.join(CLASS_NAMES)}")
print(f"\nCompression ratio: {IMG_SIZE*IMG_SIZE*3} -> {LATENT_DIM} ({IMG_SIZE*IMG_SIZE*3/LATENT_DIM:.1f}x)")

Image Size: 96x96
Batch Size: 128
Epochs: 200
Number of Classes: 10
Initial Learning Rate: 0.001

Classes: airplane, bird, car, cat, deer, dog, horse, monkey, ship, truck


## 3. Load STL-10 dataset

In [3]:
print("  Downloading STL-10 dataset...\n")


ds_unlabeled, ds_info = tfds.load(
    "stl10",
    split="unlabelled",
    as_supervised=False,
    with_info=True 
)


def ae_preprocess(example):
    image = tf.cast(example["image"], tf.float32) / 255.0
    return image, image
   


ds_unlabeled = (
    ds_unlabeled
    .map(ae_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .ignore_errors()
    .cache()
    .batch(BATCH_SIZE)
    .repeat()
    .prefetch(tf.data.AUTOTUNE)
)

unlabelled_size = ds_info.splits['unlabelled'].num_examples

print(f"  Dataset loaded successfully!")
print(f"   Number of samples: {unlabelled_size}")

print(f"   Image shape: {ds_info.features['image'].shape}")
# print(f"\n  Dataset Info:")
# print(ds_info)


  Dataset loaded successfully!
   Number of samples: 100000
   Image shape: (96, 96, 3)


I0000 00:00:1777815711.042250  396131 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46664 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:01:00.0, compute capability: 8.6


**Important:** When loading the models, using ``.repeat()`` helps prevent errors caused by corrupted samples. You must then specify the ``steps_per_epoch`` argument during training as shown below.

````
steps_per_epoch = unlabelled_size // BATCH_SIZE
autoencoder.fit(
    ds_unlabeled,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=None
)


## 4. Build the Convolutional Autoencoder

We use a symmetric convolutional autoencoder. The **encoder** compresses a 96×96×3 image (27,648 values) down to a 256-dimensional latent vector — roughly a 108:1 compression ratio.

**Architecture choices:**
- **Strided convolutions** instead of MaxPooling for downsampling — this way the network learns its own downsampling filters rather than using a fixed operation.
- **Batch normalization** after each conv layer to stabilize training and allow higher learning rates.
- **ReLU activations** throughout (standard choice for deep nets).
- **Dense bottleneck** (Flatten → Dense(256)) to get a compact 1D latent vector, which makes it easy to plug into a classifier later.
- **Sigmoid output** on the decoder since our images are normalized to [0, 1].
- The decoder mirrors the encoder using `Conv2DTranspose` (transposed convolutions) to upsample back to 96×96×3.

In [ ]:
def build_encoder(input_shape=(IMG_SIZE, IMG_SIZE, 3), latent_dim=LATENT_DIM):
    """Encoder: 96x96x3 -> latent_dim vector"""
    inputs = layers.Input(shape=input_shape, name="encoder_input")
    
    # Block 1: 96x96x3 -> 48x48x32
    x = layers.Conv2D(32, 3, strides=2, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 2: 48x48x32 -> 24x24x64
    x = layers.Conv2D(64, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 3: 24x24x64 -> 12x12x128
    x = layers.Conv2D(128, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 4: 12x12x128 -> 6x6x256
    x = layers.Conv2D(256, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Flatten and compress to latent vector
    x = layers.Flatten()(x)
    latent = layers.Dense(latent_dim, name="latent_vector")(x)
    
    encoder = models.Model(inputs, latent, name="encoder")
    return encoder

encoder = build_encoder()
encoder.summary()

In [ ]:
def build_decoder(latent_dim=LATENT_DIM):
    """Decoder: latent_dim vector -> 96x96x3 image"""
    latent_input = layers.Input(shape=(latent_dim,), name="decoder_input")
    
    # Project and reshape to spatial format: 6x6x256
    x = layers.Dense(6 * 6 * 256)(latent_input)
    x = layers.ReLU()(x)
    x = layers.Reshape((6, 6, 256))(x)
    
    # Block 1: 6x6x256 -> 12x12x128
    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 2: 12x12x128 -> 24x24x64
    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 3: 24x24x64 -> 48x48x32
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    
    # Block 4: 48x48x32 -> 96x96x3 (sigmoid for [0,1] output)
    outputs = layers.Conv2DTranspose(3, 3, strides=2, padding="same", activation="sigmoid")(x)
    
    decoder = models.Model(latent_input, outputs, name="decoder")
    return decoder

decoder = build_decoder()
decoder.summary()

In [ ]:
# Build the full autoencoder: encoder -> decoder
ae_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="ae_input")
latent = encoder(ae_input)
ae_output = decoder(latent)

autoencoder = models.Model(ae_input, ae_output, name="autoencoder")

print(f"Encoder:     {encoder.count_params():,} parameters")
print(f"Decoder:     {decoder.count_params():,} parameters")
print(f"Autoencoder: {autoencoder.count_params():,} parameters")
print(f"\nInput shape:  {autoencoder.input_shape}")
print(f"Output shape: {autoencoder.output_shape}")
print(f"Latent dim:   {LATENT_DIM}")

## 5. Train the Autoencoder

We compile the autoencoder with **MSE loss** — this is the standard choice for reconstruction since we want the output pixel values to match the input. Adam optimizer with a learning rate of 0.001.

**Callbacks:**
- **ReduceLROnPlateau**: if the loss stalls for 5 epochs, reduce the LR by 0.5. This helps escape plateaus without having to manually tune a schedule.
- **EarlyStopping**: stop training if the loss hasn't improved in 15 epochs. Prevents wasting time once the model has converged.

We use `steps_per_epoch` because of the `.repeat()` in the data pipeline (needed to handle corrupted samples gracefully).

In [ ]:
# Compile
autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss="mse"
)

# Callbacks
callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="loss",
        patience=15,
        restore_best_weights=True,
        verbose=1
    )
]

# Train
steps_per_epoch = unlabelled_size // BATCH_SIZE

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total samples: {unlabelled_size}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Starting training...\n")

history = autoencoder.fit(
    ds_unlabeled,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    callbacks=callbacks
)

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history.history["loss"], label="Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].set_title("Autoencoder Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate curve
if "lr" in history.history:
    axes[1].plot(history.history["lr"], label="Learning Rate", color="orange")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Learning Rate")
    axes[1].set_title("Learning Rate Schedule")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

print(f"\nFinal loss: {history.history['loss'][-1]:.6f}")
print(f"Best loss:  {min(history.history['loss']):.6f}")
print(f"Epochs trained: {len(history.history['loss'])}")

## 7. Visualize Reconstructions

Let's look at a few input images and their reconstructions to see how well the autoencoder learned. We don't expect pixel-perfect results with a 108:1 compression ratio, but the reconstructions should capture the overall structure and color of each image.

In [ ]:
# Grab a batch of images for visualization
sample_batch = next(iter(ds_unlabeled))
sample_images = sample_batch[0][:10]  # Take 10 images

# Get reconstructions
reconstructed = autoencoder.predict(sample_images, verbose=0)

# Plot original vs reconstructed
n = 10
fig, axes = plt.subplots(2, n, figsize=(20, 4))

for i in range(n):
    # Original
    axes[0, i].imshow(sample_images[i].numpy())
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title("Original", fontsize=12)
    
    # Reconstructed
    axes[1, i].imshow(np.clip(reconstructed[i], 0, 1))
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_title("Reconstructed", fontsize=12)

plt.suptitle("Autoencoder Reconstructions (Unlabelled Data)", fontsize=14)
plt.tight_layout()
plt.show()

# Per-image MSE
per_image_mse = np.mean((sample_images.numpy() - reconstructed) ** 2, axis=(1, 2, 3))
print(f"Per-image MSE: {per_image_mse.mean():.6f} ± {per_image_mse.std():.6f}")

## 8. Save the Encoder

We save the trained encoder so we can load it in the next notebook for latent space classification. We also save the full autoencoder in case we want to revisit it later.

In [ ]:
# Save models
encoder.save("encoder.keras")
autoencoder.save("autoencoder.keras")

print("Models saved:")
print("  - encoder.keras")
print("  - autoencoder.keras")
print(f"\nEncoder output shape: {encoder.output_shape}")
print("Ready to use for latent space classification in the next notebook.")

## AI Tool Usage Declaration

We used GitHub Copilot (integrated in VS Code) as a support tool during this lab. It was mainly used to help explore autoencoder architecture options and understand the trade-offs between different design choices (e.g., strided convolutions vs. pooling, bottleneck sizing). All architectural decisions were reviewed, understood, and justified by us before being included in the notebook.